# SPOTER WLASL100 Training
Antreneaza modelul SPOTER pentru recunoasterea semnelor ASL (100 clase).
**Activeaza GPU**: Settings → Accelerator → GPU T4

In [ ]:
import ast, json, random, warnings, copy
from typing import Optional
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings('ignore')

print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())

In [ ]:
# ── Descarca datele WLASL100 de la SPOTER ────────────────────────────────────
Path('data').mkdir(exist_ok=True)

files = {
    'data/WLASL100_train_25fps.csv': 'https://github.com/maty-bohacek/spoter/releases/download/supplementary-data/WLASL100_train_25fps.csv',
    'data/WLASL100_val_25fps.csv':   'https://github.com/maty-bohacek/spoter/releases/download/supplementary-data/WLASL100_val_25fps.csv',
}

for dest, url in files.items():
    if not Path(dest).exists():
        print(f'Downloading {Path(dest).name}...')
        urllib.request.urlretrieve(url, dest)
        print(f'  OK ({Path(dest).stat().st_size / 1024 / 1024:.1f} MB)')
    else:
        print(f'Already exists: {Path(dest).name}')

print('Date descarcate.')

In [ ]:
# ── Arhitectura SPOTER (compatibil PyTorch 2.x) ───────────────────────────────

def _get_clones(mod, n):
    return nn.ModuleList([copy.deepcopy(mod) for _ in range(n)])


class SPOTERTransformerDecoderLayer(nn.TransformerDecoderLayer):
    """Decoder layer fara self-attention (design SPOTER)."""

    def __init__(self, d_model, nhead, dim_feedforward, dropout, activation):
        super().__init__(d_model, nhead, dim_feedforward, dropout, activation)

    def forward(self, tgt, memory,
                tgt_mask=None, memory_mask=None,
                tgt_key_padding_mask=None, memory_key_padding_mask=None,
                tgt_is_causal=False, memory_is_causal=False):
        tgt = tgt + self.dropout1(tgt)
        tgt = self.norm1(tgt)
        tgt2 = self.multihead_attn(tgt, memory, memory,
                                   attn_mask=memory_mask,
                                   key_padding_mask=memory_key_padding_mask)[0]
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt


class SPOTER(nn.Module):
    """
    SPOTER - Sign POse-based TransformER.
    Input: (T, 54, 2) - T cadre, 54 landmarks, coordonate X si Y.
    """

    def __init__(self, num_classes: int, hidden_dim: int = 108):
        super().__init__()
        self.row_embed = nn.Parameter(torch.rand(50, hidden_dim))
        self.pos = nn.Parameter(
            torch.cat([self.row_embed[0].unsqueeze(0).repeat(1, 1, 1)], dim=-1)
            .flatten(0, 1).unsqueeze(0)
        )
        self.class_query = nn.Parameter(torch.rand(1, hidden_dim))
        self.transformer = nn.Transformer(hidden_dim, 9, 6, 6)
        self.linear_class = nn.Linear(hidden_dim, num_classes)

        dec = SPOTERTransformerDecoderLayer(hidden_dim, 9, 2048, 0.1, 'relu')
        self.transformer.decoder.layers = _get_clones(
            dec, self.transformer.decoder.num_layers
        )

    def forward(self, inputs):
        h = torch.unsqueeze(inputs.flatten(start_dim=1), 1).float()
        h = self.transformer(self.pos + h,
                             self.class_query.unsqueeze(0)).transpose(0, 1)
        return self.linear_class(h)


print('Model definit OK.')

In [ ]:
# ── Dataset ───────────────────────────────────────────────────────────────────

BODY = [
    'nose', 'neck', 'rightEye', 'leftEye', 'rightEar', 'leftEar',
    'rightShoulder', 'leftShoulder', 'rightElbow', 'leftElbow',
    'rightWrist', 'leftWrist',
]
HAND_BASE = [
    'wrist', 'indexTip', 'indexDIP', 'indexPIP', 'indexMCP',
    'middleTip', 'middleDIP', 'middlePIP', 'middleMCP',
    'ringTip', 'ringDIP', 'ringPIP', 'ringMCP',
    'littleTip', 'littleDIP', 'littlePIP', 'littleMCP',
    'thumbTip', 'thumbIP', 'thumbMP', 'thumbCMC',
]
HAND = [h + '_0' for h in HAND_BASE] + [h + '_1' for h in HAND_BASE]
ALL_IDS = BODY + HAND  # 54 landmarks total


class WLASL100Dataset(Dataset):
    def __init__(self, csv_path: str):
        df = pd.read_csv(csv_path, encoding='utf-8')
        df.columns = [
            c.replace('_left_', '_0_').replace('_right_', '_1_')
            for c in df.columns
        ]
        self.data, self.labels = [], []
        for _, row in df.iterrows():
            try:
                seq_len = len(ast.literal_eval(row['leftEar_X']))
                s = np.zeros((seq_len, 54, 2), dtype=np.float32)
                for j, ident in enumerate(ALL_IDS):
                    s[:, j, 0] = ast.literal_eval(row[ident + '_X'])
                    s[:, j, 1] = ast.literal_eval(row[ident + '_Y'])
                self.data.append(s)
                self.labels.append(int(row['labels']))
            except:
                continue

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.data[idx]) - 0.5
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


train_set = WLASL100Dataset('data/WLASL100_train_25fps.csv')
val_set   = WLASL100Dataset('data/WLASL100_val_25fps.csv')
print(f'Train: {len(train_set)} exemple')
print(f'Val:   {len(val_set)} exemple')
x0, y0 = train_set[0]
print(f'Shape sample: {x0.shape}, label: {y0.item()}')

In [ ]:
# ── Antrenament ───────────────────────────────────────────────────────────────

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

train_loader = DataLoader(train_set, shuffle=True)
val_loader   = DataLoader(val_set, shuffle=False)

model     = SPOTER(num_classes=100, hidden_dim=108).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.1, patience=5)

best_val_acc = 0.0
EPOCHS = 100

print(f'Antrenament SPOTER ({EPOCHS} epoci) pe {device}...')
print('-' * 55)

for epoch in range(1, EPOCHS + 1):
    # Train
    model.train()
    correct = total = 0
    for x, y in train_loader:
        x, y = x.squeeze(0).to(device), y.to(device)
        out = model(x).squeeze()
        if out.dim() == 1:
            out = out.unsqueeze(0)
        loss = criterion(out, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        pred = out.argmax(dim=-1)
        correct += (pred == y).sum().item()
        total   += y.size(0)
    train_acc = correct / total if total else 0

    # Validation
    model.eval()
    vc = vt = 0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.squeeze(0).to(device), y.to(device)
            out = model(x).squeeze()
            if out.dim() == 1:
                out = out.unsqueeze(0)
            pred = out.argmax(dim=-1)
            vc += (pred == y).sum().item()
            vt += y.size(0)
    val_acc = vc / vt if vt else 0
    scheduler.step(1 - val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'spoter_wlasl100.pth')

    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS}  train={train_acc:.3f}  val={val_acc:.3f}  best={best_val_acc:.3f}')

print('-' * 55)
print(f'GATA! Best val accuracy: {best_val_acc:.3f}')
print('Descarca fisierul spoter_wlasl100.pth din Output ->')

In [ ]:
# ── Salveaza label map ────────────────────────────────────────────────────────
WLASL_GLOSSES = {
    0: 'book', 1: 'drink', 2: 'computer', 3: 'before', 4: 'chair',
    5: 'go', 6: 'clothes', 7: 'who', 8: 'candy', 9: 'cousin',
    10: 'deaf', 11: 'different', 12: 'fine', 13: 'finish', 14: 'help',
    15: 'house', 16: 'like', 17: 'man', 18: 'many', 19: 'mother',
    20: 'name', 21: 'need', 22: 'no', 23: 'pay', 24: 'pizza',
    25: 'play', 26: 'school', 27: 'sorry', 28: 'student', 29: 'teach',
    30: 'thank', 31: 'walk', 32: 'what', 33: 'when', 34: 'where',
    35: 'who', 36: 'why', 37: 'woman', 38: 'yes', 39: 'you',
    40: 'yourself', 41: 'about', 42: 'again', 43: 'age', 44: 'all',
    45: 'any', 46: 'cool', 47: 'dark', 48: 'dog', 49: 'eat',
    50: 'father', 51: 'food', 52: 'girl', 53: 'good', 54: 'hello',
    55: 'here', 56: 'home', 57: 'hot', 58: 'know', 59: 'later',
    60: 'learn', 61: 'live', 62: 'love', 63: 'more', 64: 'never',
    65: 'now', 66: 'old', 67: 'please', 68: 'read', 69: 'right',
    70: 'see', 71: 'sister', 72: 'soon', 73: 'stop', 74: 'study',
    75: 'tell', 76: 'think', 77: 'time', 78: 'understand', 79: 'wait',
    80: 'want', 81: 'watch', 82: 'with', 83: 'work', 84: 'write',
    85: 'yesterday', 86: 'another', 87: 'bird', 88: 'black', 89: 'blue',
    90: 'brother', 91: 'buy', 92: 'clean', 93: 'come', 94: 'day',
    95: 'decide', 96: 'do', 97: 'easy', 98: 'enough', 99: 'family',
}

with open('spoter_labels.json', 'w') as f:
    json.dump(WLASL_GLOSSES, f, indent=2)

print('Fisiere generate:')
print('  spoter_wlasl100.pth  <- copiaza in backend_api/models/')
print('  spoter_labels.json   <- copiaza in backend_api/models/')